# 05 — Blogpost main flow

This notebook is the **clean tutorial notebook** for the final course deliverable.

It follows the same general style as the baseline implementation:
- explicit configuration,
- repository-aware paths,
- reproducible cached inputs,
- compact helper utilities,
- final figures exported to disk.

The goal here is **not** to rebuild the full raw-data pipeline.
Instead, we load the shipped scored trip table and the committed baseline artifact,
then produce the main figures and tables needed for the blog post.


# 0. Config and repository-aware setup


In [ ]:
%pip install xgboost

In [ ]:
from __future__ import annotations

import json
import math
import sys
import warnings
import zipfile
from itertools import combinations
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
pd.options.display.float_format = lambda x: f"{x:.4f}"
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

def find_project_root(start: Path | None = None) -> Path:
    if start is None:
        start = Path.cwd()
    start = start.resolve()

    candidates = [start] + list(start.parents)
    for cand in candidates:
        if (cand / "regression_model").exists() and (cand / "notebooks").exists():
            return cand
        if (cand / "README.md").exists() and (cand / "src").exists():
            return cand
    return start

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

# Prefer the baseline artifact because the artifact shipped inside output.zip
# can be version-sensitive across pandas / xgboost installations.
BASELINE_ARTIFACT_PATH = PROJECT_ROOT / "regression_model" / "models" / "xgb_energy_artifact.joblib"
OUTPUT_ZIP_PATH = PROJECT_ROOT / "notebooks/baseline_implementation" / "output.zip"

BLOG_OUTPUT_DIR = PROJECT_ROOT / "outputs_blogpost"
FIG_DIR = BLOG_OUTPUT_DIR / "figures"
TABLE_DIR = BLOG_OUTPUT_DIR / "tables"

for p in [BLOG_OUTPUT_DIR, FIG_DIR, TABLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RAW_NUMERIC_FEATURES = [
    "duration_min",
    "distance_km",
    "speed_mean",
    "speed_var",
    "accel_mean",
    "accel_var",
    "accel_p95",
    "stop_go_ratio",
    "idle_time_min",
    "fuel_energy_kWh",
    "battery_energy_kWh",
    "ac_energy_kWh",
    "heater_energy_kWh",
    "hv_current_abs_mean",
    "hv_current_abs_p95",
    "hv_voltage_mean",
    "maf_mean",
    "maf_p95",
    "Generalized_Weight",
]

DIRECT_TARGET_COMPONENTS = [
    "fuel_energy_kWh",
    "battery_energy_kWh",
    "ac_energy_kWh",
    "heater_energy_kWh",
]

CATEGORICAL_FEATURES = [
    "VehicleType",
    "Vehicle Class",
    "Transmission",
    "Drive Wheels",
]

TARGET = "energy_per_km"
YHAT_COL = "predicted_energy_per_km"
RESIDUAL_COL = "residual"

def load_scored_table_from_repo() -> pd.DataFrame:
    if not OUTPUT_ZIP_PATH.exists():
        raise FileNotFoundError(
            f"Could not find {OUTPUT_ZIP_PATH}. Place these notebooks into the repository root."
        )

    with zipfile.ZipFile(OUTPUT_ZIP_PATH) as zf:
        inner = "outputs_final/cache/trip_table_scored.csv.gz"
        if inner not in zf.namelist():
            raise FileNotFoundError(f"{inner} not found inside {OUTPUT_ZIP_PATH}")
        raw_gz = zf.read(inner)

    import gzip
    csv_bytes = gzip.decompress(raw_gz)
    from io import BytesIO
    df = pd.read_csv(BytesIO(csv_bytes))
    return df

def load_baseline_artifact() -> dict:
    if not BASELINE_ARTIFACT_PATH.exists():
        raise FileNotFoundError(f"Could not find baseline artifact at {BASELINE_ARTIFACT_PATH}")
    return joblib.load(BASELINE_ARTIFACT_PATH)

def safe_numeric_fill_value(s: pd.Series) -> float:
    s = pd.to_numeric(s, errors="coerce")
    median = s.median()
    if not np.isfinite(median):
        return 0.0
    return float(median)

def fill_raw_features(
        df: pd.DataFrame,
        model_numeric_features: list[str],
        categorical_features: list[str],
        numeric_fill: dict[str, float] | None = None,
    ) -> pd.DataFrame:

    selected_columns = list(model_numeric_features) + list(categorical_features)
    out = df.reindex(columns=selected_columns).copy()

    for col in model_numeric_features:
        if col not in out.columns:
            out[col] = np.nan
    for col in categorical_features:
        if col not in out.columns:
            out[col] = "NO DATA"

    if numeric_fill is None:
        numeric_fill = {col: safe_numeric_fill_value(out[col]) for col in model_numeric_features}

    for col in model_numeric_features:
        fill_value = numeric_fill.get(col, 0.0)
        if not np.isfinite(fill_value):
            fill_value = 0.0
        out[col] = pd.to_numeric(out[col], errors="coerce").fillna(fill_value)

    for col in categorical_features:
        out[col] = out[col].astype("string").fillna("NO DATA")

    return out

def make_design(
        df: pd.DataFrame,
        model_numeric_features: list[str],
        categorical_features: list[str],
        design_columns: list[str] | None = None,
        numeric_fill: dict[str, float] | None = None,
    ) -> tuple[pd.DataFrame, dict[str, float], list[str]]:

    raw = fill_raw_features(
        df,
        model_numeric_features=model_numeric_features,
        categorical_features=categorical_features,
        numeric_fill=numeric_fill,
    )
    X = pd.get_dummies(raw, columns=categorical_features, dummy_na=False)

    if design_columns is None:
        design_columns = list(X.columns)
    X = X.reindex(columns=design_columns, fill_value=0.0)

    if numeric_fill is None:
        numeric_fill = {col: safe_numeric_fill_value(raw[col]) for col in model_numeric_features}

    return X.astype(float), numeric_fill, list(design_columns)

def evaluate_regression(y_true: np.ndarray, y_pred: np.ndarray) -> dict:

    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": float(r2_score(y_true, y_pred)),
    }

def transform_target(y: np.ndarray, transform: str | None) -> np.ndarray:

    y = np.asarray(y, dtype=float)
    if transform is None:
        return y
    if transform == "log1p":
        if np.any(y < 0):
            raise ValueError("log1p requires non-negative targets")
        return np.log1p(y)
    raise ValueError(f"Unsupported transform: {transform}")

def inverse_transform_target(y: np.ndarray, transform: str | None) -> np.ndarray:

    y = np.asarray(y, dtype=float)
    
    if transform is None:
        return y
    
    if transform == "log1p":
        out = np.expm1(y)
        return np.clip(out, 0.0, None)
    
    raise ValueError(f"Unsupported transform: {transform}")


## 1. Load the shipped scored table and the baseline artifact

The scored table already contains:
- the raw trip-level features,
- the train / validation / test split,
- the model prediction,
- the residual,
- and the anomaly flag produced by the baseline pipeline.

This makes the notebook fast, reproducible, and close to the final blog-post workflow.


In [ ]:
scored_df = load_scored_table_from_repo()
artifact = load_baseline_artifact()

print("scored_df shape:", scored_df.shape)
print("artifact keys:", list(artifact.keys()))
display(scored_df.head(3))


## 2. Quick dataset and split overview

We start with a few compact summary tables that can later be reused in the blog post
and in the presentation.


In [ ]:
split_summary = (
    scored_df.groupby("split")
    .agg(
        n_trips=("trip_id", "count"),
        n_vehicles=("VehId", "nunique"),
        anomaly_rate=("is_anomaly", "mean"),
        mean_target=(TARGET, "mean"),
        mean_residual=(RESIDUAL_COL, "mean"),
    )
    .reset_index()
)

print("Split summary")
display(split_summary)

split_summary.to_csv(TABLE_DIR / "split_summary.csv", index=False)


## 3. Reconstruct the design matrix used by the baseline XGBoost model

The committed baseline artifact stores:
- `feature_columns`
- `feature_medians`
- `lime_background`

We rebuild the aligned design matrix from the raw scored table using the same feature groups
that were used in the final baseline notebook:
- exclude direct target-construction energy components from the model,
- keep behavioral and metadata features,
- one-hot encode the categorical columns.


In [ ]:
MODEL_NUMERIC_FEATURES = [c for c in RAW_NUMERIC_FEATURES if c not in DIRECT_TARGET_COMPONENTS]

X_all, numeric_fill_values, design_columns = make_design(
    scored_df,
    model_numeric_features=MODEL_NUMERIC_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
    design_columns=artifact["feature_columns"],
    numeric_fill=artifact["feature_medians"],
)

print("Aligned design matrix:", X_all.shape)
print("Number of design columns:", len(design_columns))
display(X_all.head(2))


## 4. Pipeline figure for the blog post

In [ ]:
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(13, 2.8))
ax.axis("off")

boxes = [
    (0.03, "Trip-level telematics\n(speed, acceleration,\nidling, vehicle metadata)"),
    (0.29, "XGBoost model\npredicts expected\nenergy_per_km"),
    (0.55, "Residual anomaly score\nactual - predicted"),
    (0.79, "Explain flagged trips\nwith Manual SHAP + LIME"),
]

for x, label in boxes:
    patch = FancyBboxPatch(
        (x, 0.2), 0.18, 0.58,
        boxstyle="round,pad=0.03",
        linewidth=1.5,
        edgecolor="black",
        facecolor="white",
        transform=ax.transAxes,
    )
    
    ax.add_patch(patch)
    ax.text(x + 0.09, 0.49, label, ha="center", va="center", fontsize=11, transform=ax.transAxes)

for x in [0.22, 0.48, 0.72]:
    ax.annotate(
        "",
        xy=(x + 0.05, 0.49),
        xytext=(x, 0.49),
        arrowprops=dict(arrowstyle="->", lw=1.8),
        xycoords=ax.transAxes,
        textcoords=ax.transAxes,
    )

ax.set_title("Energy anomaly detection and explanation pipeline", fontsize=13)
plt.tight_layout()
pipeline_path = FIG_DIR / "pipeline_diagram.png"
plt.savefig(pipeline_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", pipeline_path)


## 5. Predictive diagnostics

We focus on the test split because that is the cleanest part to show in the final case study.


In [ ]:
test_scored_df = scored_df.loc[scored_df["split"] == "test"].copy()

test_metrics = evaluate_regression(
    test_scored_df[TARGET].to_numpy(dtype=float),
    test_scored_df[YHAT_COL].to_numpy(dtype=float),
)

print("Test metrics:")
print(json.dumps(test_metrics, indent=2))


In [ ]:
fig = plt.figure(figsize=(6.5, 5.5))

plt.scatter(
    test_scored_df[TARGET].values,
    test_scored_df[YHAT_COL].values,
    s=10,
    alpha=0.65,
)

lims = [
    min(test_scored_df[TARGET].min(), test_scored_df[YHAT_COL].min()),
    max(test_scored_df[TARGET].max(), test_scored_df[YHAT_COL].max()),
]

plt.plot(lims, lims, linestyle="--")
plt.xlabel("Actual energy_per_km")
plt.ylabel("Predicted energy_per_km")
plt.title("Test split: actual vs predicted")
plt.tight_layout()

pred_vs_actual_path = FIG_DIR / "predicted_vs_actual_test.png"
plt.savefig(pred_vs_actual_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", pred_vs_actual_path)


In [ ]:
fig = plt.figure(figsize=(7.0, 5.0))
plt.hist(test_scored_df[RESIDUAL_COL].dropna().values, bins=60)
plt.xlabel("Residual (actual - predicted)")
plt.ylabel("Count")
plt.title("Test split: residual distribution")
plt.tight_layout()

residual_path = FIG_DIR / "residual_distribution_test.png"
plt.savefig(residual_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", residual_path)


## 6. Blog-ready summary tables

The blog post needs short, readable numbers rather than a wall of prose.
We export a compact predictive summary and a top-anomaly table.


In [ ]:
predictive_summary = pd.DataFrame([
    {
        "split": "test",
        "mae": test_metrics["mae"],
        "rmse": test_metrics["rmse"],
        "r2": test_metrics["r2"],
        "n_trips": int(len(test_scored_df)),
        "n_anomalies": int(test_scored_df["is_anomaly"].sum()),
        "anomaly_rate": float(test_scored_df["is_anomaly"].mean()),
    }
])

top_anomalies = (
    test_scored_df.loc[test_scored_df["is_anomaly"]]
    .sort_values(RESIDUAL_COL, ascending=False)
    .loc[:, ["trip_id", "VehId", TARGET, YHAT_COL, RESIDUAL_COL, "VehicleType", "Transmission", "Drive Wheels"]]
    .head(10)
    .reset_index(drop=True)
)

display(predictive_summary)
display(top_anomalies)

predictive_summary.to_csv(TABLE_DIR / "predictive_summary.csv", index=False)
top_anomalies.to_csv(TABLE_DIR / "top_test_anomalies.csv", index=False)
